# EDA rápida — pré-split (dados brutos)

Exploração mínima do `data/raw/customer_support_tickets.csv` para decidir os **parâmetros do split treino/val/teste** feito em `customer_support_analytics/dataset.py`, antes de rodá-lo.

**Objetivo:** responder só duas perguntas — (1) por qual coluna agrupar, para que nenhum cliente apareça em mais de um split, e (2) por qual coluna estratificar. Análise exploratória mais profunda (distribuições, texto, qualidade de dados etc.) fica no notebook `1.1-cb-eda-train.ipynb`, sobre o split de treino.

**Índice**
1. Carregamento dos dados
2. Estrutura básica e granularidade
3. Clientes com múltiplos tickets (chave do agrupamento)
4. Distribuição de `Ticket Priority` (coluna de estratificação)
5. Conclusão — parâmetros do split


In [1]:
import pandas as pd

from customer_support_analytics.config import RAW_DATA_DIR

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


2026-08-21 09:30:45.063 | INFO     | customer_support_analytics.config:<module>:11 - PROJ_ROOT path is: C:\Users\Micro\Documents\GitHub\estudo_customer_support


## 1. Carregamento dos dados


In [2]:
df = pd.read_csv(RAW_DATA_DIR / "customer_support_tickets.csv")
print(f"shape: {df.shape}")
df.head()


shape: (8469, 17)


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


## 2. Estrutura básica e granularidade

Confirma que cada linha é um ticket (`Ticket ID` único) — pré-requisito para qualquer split baseado em `train_test_split` sobre as linhas/grupos.


In [3]:
print(f"nº de linhas: {len(df)}")
print("Ticket ID duplicado:", df["Ticket ID"].duplicated().sum())


nº de linhas: 8469
Ticket ID duplicado: 0


## 3. Clientes com múltiplos tickets (chave do agrupamento)

Se algum `Customer Email` aparece em mais de um ticket, um split aleatório por linha poderia colocar tickets do mesmo cliente em treino **e** em val/teste (vazamento de dados). Por isso `dataset.py` agrupa por `Customer Email`.


In [4]:
tickets_per_customer = df["Customer Email"].value_counts()
repeated = tickets_per_customer[tickets_per_customer > 1]

print("clientes distintos:", df["Customer Email"].nunique(), "de", len(df), "tickets")
print("clientes com >1 ticket:", len(repeated))
print("tickets desses clientes:", repeated.sum(), f"({repeated.sum() / len(df):.1%} do total)")


clientes distintos: 8320 de 8469 tickets
clientes com >1 ticket: 139
tickets desses clientes: 288 (3.4% do total)


## 4. Distribuição de `Ticket Priority` (coluna de estratificação)

`dataset.py` estratifica pelo valor de `Ticket Priority` mais frequente de cada grupo (cliente). Vale confirmar que as classes têm volume suficiente para estratificar em 3 splits sem ficarem vazias.


In [5]:
(df["Ticket Priority"].value_counts(normalize=True) * 100).round(1)


Ticket Priority
Medium      25.9
Critical    25.1
High        24.6
Low         24.4
Name: proportion, dtype: float64

## 5. Conclusão — parâmetros do split

- **Agrupamento:** `Customer Email` — evita que o mesmo cliente apareça em mais de um split.
- **Estratificação:** `Ticket Priority` — classes com volume suficiente para manter proporções parecidas em treino/val/teste.
- **Proporções:** 70% treino / 15% val / 15% teste.

Esses são exatamente os defaults de `customer_support_analytics/dataset.py` (`group_col="Customer Email"`, `stratify_col="Ticket Priority"`). Para gerar os splits:

```
make data
```

A exploração mais profunda dos dados (distribuições, texto, qualidade dos campos de tempo etc.) continua no notebook `1.1-cb-eda-train.ipynb`, rodando **apenas sobre `data/processed/train.csv`** para não espiar val/teste.
